# Bielik 4.5B — wariant 5 epok (czysty test nasycenia)

Identyczny QLoRA jak `bielik_small`, zmienione wyłącznie `EPOCHS=3 → 5`: w runie 3-epokowym best-epoch była ostatnia epoka, więc early stopping nigdy nie zadziałał.

**Wymaga:** GPU (T4), Internet ON, dataset `pl-emotion-processed`, sekret `HF_TOKEN`.

In [ ]:
!pip install -q -U "peft>=0.12" "accelerate>=0.33" "datasets>=2.20" "bitsandbytes>=0.43" 2>/dev/null
!pip uninstall -y -q torchao 2>/dev/null   # stara torchao 0.10 psuje dispatcher LoRA w peft; nieużywana (fp16)
import torch, transformers
print("transformers", transformers.__version__, "| gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

In [ ]:
# --- HF auth z sekretu Kaggle (gated repo Bielik-4.5B) ---
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
login(UserSecretsClient().get_secret("HF_TOKEN"))
print("HF zalogowany")

In [ ]:
import os, glob, warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.special import expit
from sklearn.metrics import (f1_score, hamming_loss, jaccard_score, accuracy_score,
                             precision_score, recall_score)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer,
                          DataCollatorWithPadding, EarlyStoppingCallback)
from peft import LoraConfig, get_peft_model, TaskType
warnings.filterwarnings("ignore")

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE); np.random.seed(RANDOM_STATE)
EMOTIONS = ["radość","smutek","zaufanie","wstręt","strach","gniew","przeczuwanie","zdziwienie"]
OUT = "/kaggle/working"

# fp16 + LoRA (bez 4-bit / bitsandbytes) -> dziala na P100 i T4. Bielik 4.5B mieści się w 16 GB.
MODEL_NAME = "speakleash/Bielik-4.5B-v3.0-Instruct"
MAX_LEN, EPOCHS, BATCH, LR = 128, 5, 4, 1e-4

In [ ]:
def find_csv(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    if not hits: raise FileNotFoundError(f"{name} not in /kaggle/input — attach pl-emotion-processed")
    return hits[0]

tw_train = pd.read_csv(find_csv("twitteremo_train.csv")).reset_index(drop=True)
tw_val   = pd.read_csv(find_csv("twitteremo_val.csv")).reset_index(drop=True)
tw_test  = pd.read_csv(find_csv("twitteremo_test.csv")).reset_index(drop=True)
for df in (tw_train, tw_val, tw_test): df["tekst"] = df["tekst"].fillna("")
y_val, y_test = tw_val[EMOTIONS].values, tw_test[EMOTIONS].values
print(f"train={len(tw_train):,} val={len(tw_val):,} test={len(tw_test):,}")

In [ ]:
def evaluate(yt, yp):
    return {"f1_macro": f1_score(yt,yp,average="macro",zero_division=0),
            "f1_micro": f1_score(yt,yp,average="micro",zero_division=0),
            "f1_weighted": f1_score(yt,yp,average="weighted",zero_division=0),
            "precision_macro": precision_score(yt,yp,average="macro",zero_division=0),
            "recall_macro": recall_score(yt,yp,average="macro",zero_division=0),
            "hamming_loss": hamming_loss(yt,yp),
            "jaccard_macro": jaccard_score(yt,yp,average="macro",zero_division=0),
            "subset_accuracy": accuracy_score(yt,yp)}

def find_optimal_thresholds(yt, yp, labels):
    thr = np.full(len(labels), 0.5)
    for i in range(len(labels)):
        bf, bt = 0.0, 0.5
        for t in np.arange(0.05,0.95,0.01):
            f = f1_score(yt[:,i],(yp[:,i]>=t).astype(int),zero_division=0)
            if f>bf: bf,bt=f,t
        thr[i]=bt
    return thr

def f1_macro_ci(yt, yp, n_boot=1000, seed=RANDOM_STATE):
    rng=np.random.default_rng(seed); n=len(yt)
    base=f1_score(yt,yp,average="macro",zero_division=0)
    b=[f1_score(yt[i],yp[i],average="macro",zero_division=0) for i in (rng.integers(0,n,n) for _ in range(n_boot))]
    lo,hi=np.percentile(b,[2.5,97.5]); return base,lo,hi

In [ ]:
pos = tw_train[EMOTIONS].values.sum(0); neg = len(tw_train) - pos
POS_WEIGHT = torch.tensor(np.clip(neg/np.maximum(pos,1),1.0,10.0), dtype=torch.float32)

class WeightedTrainer(Trainer):
    def __init__(self,*a,pos_weight=None,**k): super().__init__(*a,**k); self.pos_weight=pos_weight
    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels"); out = model(**inputs); logits = out.logits
        loss = F.binary_cross_entropy_with_logits(
            logits.float(), labels.float(),
            pos_weight=self.pos_weight.to(logits.device) if self.pos_weight is not None else None)
        return (loss, out) if return_outputs else loss

In [ ]:
from transformers import BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None: tok.pad_token = tok.eos_token

def to_ds(df):
    d = Dataset.from_dict({"text": df["tekst"].tolist(),
                           "labels": df[EMOTIONS].values.astype("float32").tolist()})
    return d.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN),
                 batched=True, remove_columns=["text"])
ds_train, ds_val, ds_test = to_ds(tw_train), to_ds(tw_val), to_ds(tw_test)

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(EMOTIONS), problem_type="multi_label_classification",
    quantization_config=bnb, device_map="auto")
model.config.pad_token_id = tok.pad_token_id
model = prepare_model_for_kbit_training(model)
model.enable_input_require_grads()

lora = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
                  target_modules=["q_proj","k_proj","v_proj","o_proj"],
                  modules_to_save=["score"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
def compute_metrics(p):
    pred = (expit(p.predictions) >= 0.5).astype(int)
    return {"f1_macro": f1_score(p.label_ids.astype(int), pred, average="macro", zero_division=0)}

args = TrainingArguments(
    output_dir=f"{OUT}/bielik_ckpt", eval_strategy="epoch", save_strategy="epoch",
    save_total_limit=1, load_best_model_at_end=True, metric_for_best_model="f1_macro",
    greater_is_better=True, per_device_train_batch_size=BATCH, per_device_eval_batch_size=8,
    gradient_accumulation_steps=4, gradient_checkpointing=True, num_train_epochs=EPOCHS,
    learning_rate=LR, warmup_ratio=0.1, weight_decay=0.01, fp16=True, logging_steps=50,
    report_to="none", seed=RANDOM_STATE, optim="adamw_torch")

trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
    data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics,
    pos_weight=POS_WEIGHT, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
trainer.train()

In [ ]:
p_val  = expit(trainer.predict(ds_val).predictions)
p_test = expit(trainer.predict(ds_test).predictions)
thr = find_optimal_thresholds(y_val, p_val, EMOTIONS)
pred_test = (p_test >= thr).astype(int)
m = evaluate(y_test, pred_test)
base, lo, hi = f1_macro_ci(y_test, pred_test)
m.update({"model": MODEL_NAME.split("/")[-1]+"-5ep", "ci_low": round(lo,3), "ci_high": round(hi,3)})
res = pd.DataFrame([m])
res.to_csv(f"{OUT}/bielik45_5ep_results.csv", index=False)
np.save(f"{OUT}/bielik45_5ep_proba_test.npy", p_test)
np.save(f"{OUT}/bielik45_5ep_proba_val.npy", p_val)  # val -> strojenie progow ensemble (notebook 15)
print(f"F1-Macro={m['f1_macro']:.3f}  [{lo:.3f}, {hi:.3f}]  F1-Micro={m['f1_micro']:.3f}")
res[["model","f1_macro","ci_low","ci_high","f1_micro","jaccard_macro","hamming_loss"]]

## Wynik
`/kaggle/working/bielik45_results.csv` + `bielik45_proba_test.npy` (prawdopodobieństwa do ensemble).